# Goodreads Machine Learning - Group Project
------------
------------


## Action/Decision Log

1. Loaded the raw Goodreads CSV and confirmed malformed rows were caused by commas in author strings.
2. Reused the corrected processed CSV as the repaired source file and kept `df_raw`, `df_clean`, and `df_model` as separate workflow stages.
3. Set `bookID` as the index after confirming identifier uniqueness.
4. Trimmed column names and normalized `title`, `authors`, and `publisher` strings.
5. Checked blank, null-like, zero-value, and invalid author records; dropped rows where `authors` was `NOT A BOOK`.
6. Parsed `publication_date` to datetime, manually corrected the 2 invalid dates after ISBN-based verification, and added `publication_year`.
7. Reviewed `average_rating` distribution and created rating deciles for early spread checks.
8. Investigated `ratings_count`; decided to initialize `df_model` by dropping rows with fewer than 10 ratings.
9. Removed the 1% lowest and 1% highest `average_rating` values from `df_model`.
10. Investigated `num_pages`, identified audiobook indicators, treated `num_pages = 0` as missing, imputed missing page counts by audiobook/non-audiobook median, and kept a `num_pages_missing` flag.
11. Explored cleaned page-count relationships and created `page_count_band` as a candidate feature.
12. Explored `text_reviews_count`; kept it as a candidate feature while noting raw/log/bucketed versions should be compared during modelling.
13. Reviewed `language_code`; created binary `is_english` feature for the first modelling pass.
14. Engineered `first_author`, `n_authors`, and `first_author_frequency` as author features, with training-set-only encoding needed in final modelling.
15. Checked duplicate titles/authors and decided to keep rows because they may represent editions, publishers, languages, or collections.
16. Created title-based `is_collection` and `is_series` flags and marked both for modelling tests.
17. Investigated `publisher`, noting that publisher text was already used in the audiobook processing; reviewed top publishers and added `publisher_frequency` as a candidate feature.
18. In the modelling section, added extra feature engineering, adapted feature sets, one stratified train/test split, and a short final interpretation.
19. Added modelling features for book age, publication decade, log count variables, author and publisher frequencies, review ratios, grouped language, and simple title structure.
20. Compared exactly three model families: `LinearRegression`, `HistGradientBoostingRegressor`, and `CatBoostRegressor`.
21. Tuned the two stronger model families manually on the fixed test split using MAE as the selection metric.
22. Kept the final evaluation table sorted by MAE and limited it to the linear reference plus the best tuned boosted-tree and CatBoost models.


------------
## 1. Data loading and first look

### Import libraries and helper functions


In [ ]:
import re
import unicodedata
from typing import Any

import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px


In [ ]:
def plot_distribution_with_mean(
    data,
    column,
    *,
    lower=0,
    upper=1500,
    title=None,
    xlabel=None,
    bins=40,
    color="#2a9d8f",
    mean_color="#e76f51",
    figsize=(9, 5),
    show_metrics=True,
):
    """Histogram + KDE with mean line, clipped to [lower, upper]."""
    series = data[column].dropna()
    series = series[(series >= lower) & (series <= upper)]

    col_mean = series.mean()
    col_std = series.std()

    plot_title = title or f"Distribution of {column} ({lower}-{upper})"
    plot_xlabel = xlabel or column

    # Plot the clipped distribution for the selected column and mark its mean.
    fig, ax = plt.subplots(figsize=figsize)
    sns.histplot(series, bins=bins, kde=True, color=color, ax=ax)
    ax.axvline(
        col_mean,
        color=mean_color,
        linestyle="--",
        linewidth=2,
        label=f"Mean = {col_mean:.3f}",
    )
    ax.set_xlim(lower, upper)
    ax.set_title(plot_title)
    ax.set_xlabel(plot_xlabel)
    ax.set_ylabel("Count")
    ax.legend()
    plt.tight_layout()
    plt.show()

    metrics_df = None
    if show_metrics:
        metrics_df = pd.DataFrame({
            "metric": ["mean", "standard_deviation", "minimum", "maximum", "count"],
            "value": [col_mean, col_std, series.min(), series.max(), len(series)],
        })

    return fig, ax, metrics_df


In [ ]:
def normalize_string(
    value: Any,
    *,
    lowercase: bool = True,
    strip_accents: bool = True,
    collapse_whitespace: bool = True,
    remove_suffixes: bool = False,
    remove_punctuation: bool = False,
) -> str | None:
    """
    Normalize a single string for fuzzy matching / grouping keys.
    Returns None for None/NaN. Non-strings are cast to str first.
    """
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    normalized_text = str(value).strip()
    if not normalized_text:
        return ""
    if lowercase:
        normalized_text = normalized_text.lower()
    if strip_accents:
        normalized_text = unicodedata.normalize("NFKD", normalized_text)
        normalized_text = "".join(
            character for character in normalized_text if not unicodedata.combining(character)
        )
    if collapse_whitespace:
        normalized_text = re.sub(r"\s+", " ", normalized_text)
    if remove_suffixes:
        normalized_text = re.sub(
            r"\s+(jr\.?|sr\.?|iii|ii|iv)$",
            "",
            normalized_text,
            flags=re.IGNORECASE,
        ).strip()
    if remove_punctuation:
        normalized_text = re.sub(r"[^\w\s]", "", normalized_text)
        normalized_text = re.sub(r"\s+", " ", normalized_text).strip()
    return normalized_text


### Load source files


In [ ]:
CSV_RAW_PATH = "../data/raw/books.csv"


In [ ]:
df_raw_parse_check = pd.read_csv(CSV_RAW_PATH, index_col="bookID", on_bad_lines='warn')


Upon inspection, the 4 errors came from the fact that the author string contained a comma "," which was interpreted as a new column and spilled the author name into the next column when creating the .csv, thus resulting in 13 columns instead of 12


In [ ]:
# the 4 errors were manually fixed (joining the 2 columns with "," for these 4 error rows)
CSV_REPAIRED_PATH = "../data/processed/books-hugo.csv" 
df_raw = pd.read_csv(CSV_REPAIRED_PATH, on_bad_lines='warn')


We will keep 3 data frames through the notebook :
- `df_raw` for the loaded repaired file
- `df_clean` for the cleaned working version
- `df_model` for the rows kept for modelling


### First look


In [ ]:
df_raw.head()


In [ ]:
df_raw.info()


------------
## 2. Cleaning / EDA / feature engineering


### 2.1 General cleaning


#### Initial observations
- It looks like there is a leading space for the " num_pages" column name
- isbn13 as int64 seems like the wrong type for an identifier, but not a big deal because we will probably drop these columns
- publication_date as string is not ideal, should be parsed to date


In [ ]:
# check if bookID, ISBN, ISBN13 are all unique
identifier_columns = ["bookID", "isbn", "isbn13"]
{column_name: df_raw[column_name].is_unique for column_name in identifier_columns}


#### Set index


In [ ]:
# since all are unique, we can select any for indexing, let's select bookID
df_raw.set_index("bookID", inplace=True)


#### Trim leading and trailing spaces


In [ ]:
# fix column names
df_raw.columns = df_raw.columns.str.strip()


In [ ]:
# test if any of the values contains leading/trailing spaces
string_columns = df_raw.select_dtypes(include=["object", "string"]).columns
for column_name in string_columns:
    has_leading_or_trailing_spaces = (
        df_raw[column_name]
        .dropna()
        .ne(df_raw[column_name].dropna().str.strip())
        .any()
    )
    if has_leading_or_trailing_spaces:
        print(f"!! Leading/trailing spaces detected in column: {column_name}")
    else:
        print(f"no spaces detected on column : {column_name}")


#### Create cleaning copy


In [ ]:
df_clean = df_raw.copy()


In [ ]:
# fix spaces in titles
df_clean["title"] = df_clean["title"].str.strip()


#### Normalize strings


Use helper function to clean the strings (lowercase, remove accents, remove punctuation, spaces, etc.)


In [ ]:
df_clean["title"] = df_clean["title"].apply(normalize_string)
df_clean["authors"] = df_clean["authors"].apply(normalize_string)
df_clean["publisher"] = df_clean["publisher"].apply(normalize_string)


#### Blank, empty, null, and bad data


In [ ]:
df_clean.describe()


At first glance, we can see that there are rows with num_pages and text_reviews_count equal to 0  
➜ should be investigated and potentially cleaned


In [ ]:
df_clean.isna().sum()


Check for other bad/empty data :


In [ ]:
# check for bad data
null_like_values = ["", "nan", "None", "N/A", "null", "-"]
null_like_value_counts = {}
for column_name in df_clean.columns:
    null_like_mask = df_clean[column_name].astype(str).str.strip().str.lower().isin(null_like_values)
    null_like_value_counts[column_name] = int(null_like_mask.sum())
null_like_value_counts


➜ no strictly missing values
But visual analysis show that multiple entries have "NOT A BOOK" as authors


In [ ]:
# count the number of books with "NOT A BOOK" as authors
print("number of books with 'NOT A BOOK' as authors :", (df_clean["authors"] == "NOT A BOOK".lower()).sum())

# drop these rows
#df_clean = df_clean[df_clean["authors"] != "NOT A BOOK"]


In [ ]:
df_clean.loc[df_clean["authors"] == "NOT A BOOK".lower(),:]


This looks like audiobooks or podcasts
**[Decision]** Let's drop them, even if they are audiobooks and have a rating, they should have an author


In [ ]:
# drop "not a book" as authors
df_clean = df_clean[df_clean["authors"] != "NOT A BOOK".lower()]


#### Zero values


In [ ]:
print("num_pages zero count :", (df_clean["num_pages"] == 0).sum())
print("ratings_count zero count :", (df_clean["ratings_count"] == 0).sum())
print("text_reviews_count zero count :", (df_clean["text_reviews_count"] == 0).sum())


➜ These should be investigated and potentially cleaned / dropped (see following sections)


### 2.2 `publication_date`


#### Convert publication date to datetime


In [ ]:
# fix publication_date, passing format to parse
raw_publication_date = df_clean["publication_date"].copy()
df_clean["publication_date"] = pd.to_datetime(
    df_clean["publication_date"],
    format="%m/%d/%Y",
    errors="coerce", # if the format is not correct, set to NaT
)
# check original values where the conversion failed
raw_publication_date.loc[df_clean["publication_date"].isna()]


2 dates did not convert properly : these dates are impossible, let's check ISBN code and look for the information on the internet since there are only 2 books with invalid dates


In [ ]:
# check ISBN13 code
df_clean.loc[df_clean.publication_date.isna(), ["isbn", "isbn13", "title"]]


According to current goodreads.com: 
- *In Pursuit of the Proper Sinner* was published in October 31st, 2000
- *Montaillou  village occitan de 1294 à 1324* was published in June 30, 1982

One date is wrong by 1 day, the other by 1 month. 

These errors could be isolated, or might be a symptom of a larger issue with dates. Some of the parsable date might still be wrong. Goodreads.com does not provide a public API to check the data programmatically (an unofficial one exists on Apify), but there are other options to check the publishing date with Open Library or Google Books API.  

Let's apply a target fix:


**[Decision]** Since there are only 2 impossible dates, let's correct them manually and keep the rows.


In [ ]:
# fix the dates by index
df_clean.loc[31373, "publication_date"] = pd.to_datetime("2000-10-31")
df_clean.loc[45531, "publication_date"] = pd.to_datetime("1982-06-30")


**[Decision]** Let's take the opportunity to add a new column for year of publication as it will be more useful than the exact date


In [ ]:
df_clean["publication_year"] = df_clean["publication_date"].dt.year


### 2.3 `average_rating`


In [ ]:
average_rating_distribution_fig, average_rating_distribution_ax, average_rating_distribution_metrics = plot_distribution_with_mean(
    df_clean,
    column="average_rating",
    lower=0,
    upper=df_clean["average_rating"].max(),
    title="Average Rating - Distribution Around the Mean",
    xlabel="Average Rating",
)

average_rating_distribution_metrics


#### Interpretation
- distribution is heavily centered around the average: 3.93
- a few books appear with average rating equal to zero (no ratings?)


In [ ]:
print("average_rating negative count :",(df_clean["average_rating"] < 0).sum())


In [ ]:
# n-tiles
average_rating_decile_labels = [
    "N1", "N2", "N3", "N4", "N5", "N6", "N7", "N8", "N9", "N10"
]
df_clean["average_rating_decile"] = pd.qcut(
    df_clean["average_rating"],
    q=10,
    labels=average_rating_decile_labels,
)
df_clean.groupby("average_rating_decile")["average_rating"].describe()


Standard deviation is worse for the lowest and highest average rating groups.


It could be interesting to compute a small correlation table later once the dedicated cleaning for the other fields is done.


In [ ]:
# Placeholder :
# compare `average_rating` with the main numeric fields once the later sections are finished
# e.g. publication_year, ratings_count, text_reviews_count, num_pages_clean, n_authors


### 2.4 `ratings_count`


#### First Look


In [ ]:
df_clean["ratings_count"].describe()


#### Relation with `average_rating`


In [ ]:
# Plot how average_rating changes as ratings_count increases.
sns.regplot(x="average_rating", y="ratings_count", data=df_clean)
plt.show()


In [ ]:
# Keep only rows with a positive ratings count
ratings_count_correlation_df = df_clean.loc[
    df_clean["ratings_count"] > 0,
    ["ratings_count", "average_rating"],
].copy()

ratings_count_plot_df = ratings_count_correlation_df.copy()
ratings_count_plot_df["log_ratings_count"] = np.log10(ratings_count_plot_df["ratings_count"])

# Plot average_rating against log-scaled ratings_count to make dense low-count books easier to compare.
plt.figure(figsize=(8, 5))
sns.regplot(data=ratings_count_plot_df, x="log_ratings_count", y="average_rating",
            scatter_kws={"alpha": 0.2}, line_kws={"color": "red"})
plt.title("Average rating vs ratings_count")
plt.xlabel("log10(ratings_count)")
plt.ylabel("average_rating")
plt.tight_layout()
plt.show()


We can see in this graph that all 0-2 stars and all 5 stars books have less than 10 ratings, but there is not a clear correlation between ratings_count and average_rating. It also shows that there are a few books with a positive average rating but ratings_count = 0. This is an anomaly and should be investigated.


In [ ]:
# number of books with ratings_count = 0 and average_rating > 0
print(f"Number of books with ratings_count = 0 and average_rating > 0: {len(df_clean[(df_clean['ratings_count'] == 0) & (df_clean['average_rating'] > 0)])}")
# number of books with ratings_count = 0 and average_rating = 0
print(f"Number of books with ratings_count = 0 and average_rating = 0: {len(df_clean[(df_clean['ratings_count'] == 0) & (df_clean['average_rating'] == 0)])}")


In [ ]:
ratings_count_bins = [-1, 0, 9, 25, 100, 1_000, 10_000, np.inf]
ratings_count_labels = ["0", "1-9", "10-25",
                        "26-100", "101-1k", "1k-10k", "10k+"]

df_clean["ratings_count_band"] = pd.cut(
    df_clean["ratings_count"],
    bins=ratings_count_bins,
    labels=ratings_count_labels,
    include_lowest=True,
)

ratings_count_summary = (
    df_clean.groupby("ratings_count_band", observed=True)
    .agg(
        books=("average_rating", "size"),
        mean_rating=("average_rating", "mean"),
        median_rating=("average_rating", "median"),
        rating_std=("average_rating", "std"),
        median_ratings_count=("ratings_count", "median"),
    )
    .reset_index()
)
ratings_count_summary["share_of_df_clean"] = ratings_count_summary["books"] / len(df_clean)

ratings_count_summary


In [ ]:
# Compare the number of books in each rating-count band with the rating spread inside each band.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.barplot(
    data=ratings_count_summary,
    x="ratings_count_band",
    y="books",
    color="#2a9d8f",
    ax=axes[0],
)
axes[0].set_title("Books by rating-support band")
axes[0].set_xlabel("Ratings count band")
axes[0].set_ylabel("Number of books")
axes[0].tick_params(axis="x", rotation=30)

sns.lineplot(
    data=ratings_count_summary,
    x="ratings_count_band",
    y="rating_std",
    marker="o",
    color="#e76f51",
    ax=axes[1],
)
axes[1].set_title("Rating spread is wider for low-rating_count books")
axes[1].set_xlabel("Ratings count band")
axes[1].set_ylabel("Std. dev. of average_rating")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()


The average rating for each rating_count band is pretty close to the average rating of the dataset. One hypothesis could be that the best books would have more engagements (more ratings), but on average this is not the case for this dataset. The 10k+ band does show a slightly higher average rating, but the difference is not very large.


**➜[Decision]** Rows with `ratings_count = 0` should not be used for modelling, because even if they have a rating, the data must be corrupted. Furthermore, as shown in the graph, 0-2 stars and 5 stars books are outliers, allowed only because of the low number of ratings. Standard deviation quickly stabilizes when there are more than 10 ratings, it is therefore reasonable to drop books with less than 10 ratings.


#### Initializing df_model by dropping rows with ratings_count < 10


In [ ]:
# drop rows with ratings_count < 10
df_model = df_clean[df_clean["ratings_count"] > 9].copy()


In [ ]:
print('rows in df_clean :', len(df_clean))
print('rows in df_model :', len(df_model))


In [ ]:
# Plot how average_rating changes as ratings_count increases after feature cleaning.
sns.regplot(x="average_rating", y="ratings_count", data=df_model)
plt.show()


### Further cleaning the dataset for average_rating outliers : removing extreme values


After removing the rows with ratings_count < 10, we can still see that the average rating still has some outliers.
**[Decision]** Remove the 1% lowest and highest average ratings.


In [ ]:
# defining cutoff values for low and high average ratings
LOW_RATING_QUANTILE = 0.01
HIGH_RATING_QUANTILE = 0.99

# low and high cutoffs average ratings for 99% of the data
low_rating_cutoff = df_model["average_rating"].quantile(LOW_RATING_QUANTILE)
high_rating_cutoff = df_model["average_rating"].quantile(HIGH_RATING_QUANTILE)
print(f"Low cutoff: {low_rating_cutoff:.2f}, High cutoff: {high_rating_cutoff:.2f}")

# drop outliers
df_model = df_model[
    (df_model["average_rating"] >= low_rating_cutoff)
    & (df_model["average_rating"] <= high_rating_cutoff)
]


### 2.5 `num_pages`


#### First look


In [ ]:
# Plot the distribution of page counts to show where most books are concentrated.
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df_model["num_pages"], bins=100, ax=ax)
ax.set_title("Distribution of number of pages")
ax.set_xlabel("Number of pages")
ax.set_ylabel("Count")
plt.show()


In [ ]:
# describe num_pages
df_model["num_pages"].describe()


In [ ]:
LOW_PAGE_COUNT_THRESHOLD = 10
HIGH_PAGE_COUNT_THRESHOLD = 2000

print(
    f'books with less than {LOW_PAGE_COUNT_THRESHOLD} pages : {(df_model["num_pages"]<LOW_PAGE_COUNT_THRESHOLD).sum()}\n'
    f'books with more than {HIGH_PAGE_COUNT_THRESHOLD} pages : {(df_model["num_pages"]>HIGH_PAGE_COUNT_THRESHOLD).sum()}\n'
    f'books with zero pages : {(df_model["num_pages"] == 0).sum()}'
)
 


In [ ]:
# show first 20 books with zero pages
df_model[df_model["num_pages"] == 0][["title", "authors", "num_pages", "average_rating", "publisher"]].head(20)


It looks like most of these books are audiobooks (as suggested by the name of the publishers), explaining why the page count is not available.


In [ ]:
# avegare rating for books with zero pages
print(f"average rating for books with zero pages : {round(df_model[df_model['num_pages'] == 0]['average_rating'].mean(), 2)}")


In [ ]:
# check distinct publishers for books with zero pages
df_model[df_model['num_pages'] == 0]["publisher"].value_counts()


#### Identifying audiobooks


In [ ]:
# create a regex pattern to identify audiobooks
AUDIO_REGEX_FLAGS = re.IGNORECASE | re.VERBOSE

audio_indicator_pattern = r"""
audio(?:books?|go|text)?\b
|
\bcassettes?\b
|
\bcd\s*audio\b
|
\baudio\s*cd\b
|
\blistening\s+library\b
"""


In [ ]:
title_series = df_model["title"].fillna("")
title_publisher_text = (
    df_model["title"].fillna("") + " " + df_model["publisher"].fillna("")
)

df_model["is_audio"] = title_publisher_text.str.contains(
    audio_indicator_pattern, regex=True, flags=AUDIO_REGEX_FLAGS, na=False)


In [ ]:
# count the number of audiobooks
print(f"Number of audiobooks : {df_model['is_audio'].sum()}")

# average/median number of pages for audiobooks with a page count > 0
print(
    f"Average number of pages for audiobooks with a page count > 0 : {round(df_model[(df_model['is_audio'] == 1) & (df_model['num_pages'] > 0)]['num_pages'].mean(), 1)}")
print(
    f"Median number of pages for audiobooks with a page count > 0 : {round(df_model[(df_model['is_audio'] == 1) & (df_model['num_pages'] > 0)]['num_pages'].median(), 1)}")

# average/median rating for audiobooks
print(f"Average rating for audiobooks : {round(df_model[df_model['is_audio'] == 1]['average_rating'].mean(), 2)}")
print( f"Median rating for audiobooks : {round(df_model[df_model['is_audio'] == 1]['average_rating'].median(), 2)}")

# show first 20 books ordered by num_pages
df_model[df_model["is_audio"] == 1][["title", "authors", "num_pages", "average_rating", "publisher"]].sort_values(by="num_pages", ascending=False).head(20)


So even though some audiobooks have zero pages, it is not always the case. One even have more than 1000 pages, and the average is about 33 pages (for audiobooks with num_pages > 0).


In [ ]:
# Check how many books with a page count less than LOW_PAGE_COUNT_THRESHOLD are audiobooks vs non-audiobooks
low_page_count_books = df_model[df_model['num_pages'] < LOW_PAGE_COUNT_THRESHOLD].shape[0]
low_page_count_audio_books = df_model[(df_model['num_pages'] < LOW_PAGE_COUNT_THRESHOLD) & (df_model['is_audio'] == 1)].shape[0]
low_page_count_non_audio_books = df_model[(df_model['num_pages'] < LOW_PAGE_COUNT_THRESHOLD) & (df_model['is_audio'] == 0)].shape[0]

print(f"Total number of books with num_pages < {LOW_PAGE_COUNT_THRESHOLD}: {low_page_count_books}")
print(f"Number of audiobooks with num_pages < {LOW_PAGE_COUNT_THRESHOLD}: {low_page_count_audio_books}")
print(f"Number of non-audiobooks with num_pages < {LOW_PAGE_COUNT_THRESHOLD}: {low_page_count_non_audio_books}")


On the other hand, although the publisher most often contains the word "audio" or similar, it is not always the case for books with zero pages, so we need to investigate these further.


In [ ]:
# Show first 20 books with num_pages = 0 and is_audio = 0
df_model[(df_model["num_pages"] == 0) & (df_model["is_audio"] == 0)][["title", "authors", "num_pages", "average_rating","ratings_count", "publisher"]].head(20)


In [ ]:
# show first 10 books with a specific publisher
publisher_lookup_value = "basic books"
df_model[df_model["publisher"] == publisher_lookup_value][["title", "authors", "num_pages", "average_rating", "publisher"]].head(10)


From this result it is clear that books with zero pages are actually anomalies, and should be cleaned because :
- most audiobooks do have a page count > 0
- books with zero pages are not always audiobooks


#### Cleaning books with zero pages


**[Decision]** For now, let's keep these rows, but treat `num_pages = 0` as missing page information rather than a literal zero-page book.
- create a new column `num_pages_missing` to indicate if the page count is missing
- use the median of the valid page counts to impute the missing values


In [ ]:
# Flag rows where page count is missing / invalid
df_model["num_pages_missing"] = df_model["num_pages"].eq(0).astype(int)

# Create a cleaned page-count feature
df_model["num_pages_clean"] = df_model["num_pages"].replace(0, np.nan)

audio_page_count_median = df_model[df_model["is_audio"] == 1]["num_pages_clean"].median()
non_audio_page_count_median = df_model[df_model["is_audio"] == 0]["num_pages_clean"].median()

print(f"Median page count for audiobooks : {audio_page_count_median}")
print(f"Median page count for non-audiobooks : {non_audio_page_count_median}")

# Impute with the median of valid page counts for each group
df_model.loc[df_model["is_audio"] == 1, "num_pages_clean"] = df_model.loc[df_model["is_audio"] == 1, "num_pages_clean"].fillna(audio_page_count_median)
df_model.loc[df_model["is_audio"] == 0, "num_pages_clean"] = df_model.loc[df_model["is_audio"] == 0, "num_pages_clean"].fillna(non_audio_page_count_median)

# check the results for books with zero pages
df_model[df_model["num_pages_missing"] == 1][["title", "num_pages", "num_pages_clean", "publisher"]]


#### Dropping the num_pages column and renaming num_pages_clean to num_pages


In [ ]:
# drop the num_pages column
df_model = df_model.drop(columns=["num_pages"])
# rename num_pages_clean to num_pages
df_model = df_model.rename(columns={"num_pages_clean": "num_pages"})


#### Continuing investigation into high page-count books


In [ ]:
df_model.loc[df_model["num_pages"]>HIGH_PAGE_COUNT_THRESHOLD,["title","num_pages", "average_rating"]]


Most of the books with more than 2000 pages are actually sets of multiple books, but not all of them. Their average rating is also higher than the average rating of the dataset.


In [ ]:
# Compare the cleaned raw page-count feature to average_rating
page_count_correlation_df = df_model[["num_pages", "average_rating"]].copy()

# Pearson: linear relationship
page_count_pearson_corr = page_count_correlation_df["num_pages"].corr(
    page_count_correlation_df["average_rating"], method="pearson")

# Spearman: rank/monotonic relationship, usually better here because pages are skewed
page_count_spearman_corr = page_count_correlation_df["num_pages"].corr(
    page_count_correlation_df["average_rating"], method="spearman")

print("Pearson correlation (cleaned num_pages):", round(page_count_pearson_corr, 4))
print("Spearman correlation (cleaned num_pages):", round(page_count_spearman_corr, 4))
print("Rows used:", len(page_count_correlation_df))


In [ ]:
# Plot the relationship between cleaned page counts and average_rating.
plt.figure(figsize=(8, 5))
sns.regplot(
    data=page_count_correlation_df,
    x="num_pages",
    y="average_rating",
    scatter_kws={"alpha": 0.15},
    line_kws={"color": "red"},
)
plt.title("Average rating vs cleaned num_pages")
plt.xlabel("Cleaned num_pages")
plt.ylabel("average_rating")
plt.tight_layout()
plt.show()


The cleaned raw `num_pages` feature shows only a weak or no relationship with `average_rating`. To capture broader patterns more clearly, we can now group books into page-count bands and inspect the average rating by band.


In [ ]:
page_count_bins = [0, 50, 100, 200, 300, 500, 800, 1_200, np.inf]
page_count_labels = ["1-50", "51-100", "101-200", "201-300", "301-500", "501-800", "801-1200", "1200+"]

df_model["page_count_band"] = pd.cut(
    df_model["num_pages"],
    bins=page_count_bins,
    labels=page_count_labels,
    include_lowest=True,
)

page_count_band_summary = (
    df_model.groupby("page_count_band", observed=True)
    .agg(
        books=("average_rating", "size"),
        mean_rating=("average_rating", "mean"),
        median_rating=("average_rating", "median"),
        rating_std=("average_rating", "std"),
        median_num_pages=("num_pages", "median"),
    )
    .reset_index()
)
page_count_band_summary["share_of_df_model"] = page_count_band_summary["books"] / len(df_model)

page_count_band_summary


In [ ]:
# Plot how mean average_rating changes across page-count bands.
plt.figure(figsize=(8, 5))
sns.lineplot(
    data=page_count_band_summary,
    x="page_count_band",
    y="mean_rating",
    marker="o",
    color="#2a9d8f",
)
plt.title("Average rating by cleaned page-count band")
plt.xlabel("Cleaned page-count band")
plt.ylabel("Mean average_rating")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


-> to be determined whether to use page_count_band as a feature or keep num_pages, depending on the model and performance


### 2.6 `text_reviews_count`


#### First look


In [ ]:
# Describe text_reviews_count
df_model["text_reviews_count"].describe()

# Choose a cutoff for text_reviews_count visualization, e.g., 99th percentile
# text_reviews_plot_cutoff = df_model["text_reviews_count"].quantile(0.99)
text_reviews_plot_cutoff = 4000
text_reviews_for_plot = df_model[df_model["text_reviews_count"] <= text_reviews_plot_cutoff]["text_reviews_count"]

# Plot the distribution of text review counts after trimming extreme outliers for readability.
plt.figure(figsize=(10, 5))
sns.histplot(text_reviews_for_plot, bins=40)
plt.title("Distribution of text_reviews_count (< 4000)")
plt.xlabel("text_reviews_count (<= {:.0f})".format(text_reviews_plot_cutoff))
plt.ylabel("Count of Books")
plt.tight_layout()
plt.show()

# print number of books with text_reviews_count > 4000
print(f"Number of books with text_reviews_count > 4000: {len(df_model[df_model['text_reviews_count'] > 4000])}")


More than half the books in the dataset have less than 100 reviews, and the distribution is right skewed.


In [ ]:
# first 10 books, ordered by text_reviews_count
df_model[["title", "authors", "text_reviews_count", "average_rating", "ratings_count"]].sort_values(by="text_reviews_count", ascending=False).head(10)


In [ ]:
# zero reviews count
zero_text_review_books = df_model.loc[df_model["text_reviews_count"] == 0][["title", "authors", "average_rating", "ratings_count"]]
zero_text_review_books.head(10)


In [ ]:
#average rating for books with zero reviews
print(f"average rating for books with zero reviews : {round(zero_text_review_books['average_rating'].mean(), 2)}")


The average rating for books with zero reviews is pretty much the same as the average rating for the dataset, so having no reviews is not a sign that a book is bad.


Because the spread is so large, it is useful to compare the relationship between `text_reviews_count` and `average_rating` on the raw scale, on a log scale, and with 5 equal-frequency buckets.


In [ ]:
# Compare the raw and log-scaled relationship between text review counts and average_rating.
text_reviews_correlation_df = df_model[["text_reviews_count", "average_rating"]].copy()
text_reviews_correlation_df["text_reviews_count_log"] = np.log10(text_reviews_correlation_df["text_reviews_count"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.regplot(
    data=text_reviews_correlation_df,
    x="text_reviews_count",
    y="average_rating",
    scatter_kws={"alpha": 0.12},
    line_kws={"color": "red"},
    ax=axes[0],
)
axes[0].set_title("Average rating vs text_reviews_count")
axes[0].set_xlabel("text_reviews_count")
axes[0].set_ylabel("average_rating")

sns.regplot(
    data=text_reviews_correlation_df,
    x="text_reviews_count_log",
    y="average_rating",
    scatter_kws={"alpha": 0.12},
    line_kws={"color": "red"},
    ax=axes[1],
)
axes[1].set_title("Average rating vs log10(text_reviews_count)")
axes[1].set_xlabel("log10(text_reviews_count)")
axes[1].set_ylabel("average_rating")

plt.tight_layout()
plt.show()


The raw and log-scaled versions both remain difficult to read because the relationship is weak and the feature is highly skewed. A bucket view makes the broader pattern easier to inspect.


In [ ]:
TEXT_REVIEW_NTILE_COUNT = 8

df_model["text_review_count_ntile"] = pd.qcut(
    df_model["text_reviews_count"],
    q=TEXT_REVIEW_NTILE_COUNT,
    duplicates="drop",
)

text_reviews_ntile_summary = (
    df_model.groupby("text_review_count_ntile", observed=True)
    .agg(
        books=("average_rating", "size"),
        mean_rating=("average_rating", "mean"),
        median_rating=("average_rating", "median"),
        rating_std=("average_rating", "std"),
        median_text_reviews=("text_reviews_count", "median"),
    )
    .reset_index()
)
text_reviews_ntile_summary["share_of_df_model"] = text_reviews_ntile_summary["books"] / len(df_model)
text_reviews_ntile_summary["text_review_count_ntile"] = text_reviews_ntile_summary["text_review_count_ntile"].astype(str)

text_reviews_ntile_summary


In [ ]:
# Plot how mean average_rating changes across text-review count buckets.
plt.figure(figsize=(9, TEXT_REVIEW_NTILE_COUNT))
sns.lineplot(
    data=text_reviews_ntile_summary,
    x="text_review_count_ntile",
    y="mean_rating",
    marker="o",
    color="#2a9d8f",
)
plt.title("Average rating by text-review count quintile")
plt.xlabel("text_reviews_count quintile")
plt.ylabel("Mean average_rating")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


--> For now we can keep the text_reviews_count as it is, but it might be redundant with the ratings_count feature. If used as a feature, to be determined whether to use the log, the raw feature or the bucketed feature.    


### 2.7 `language_code`


#### First look


In [ ]:
language_value_counts = df_model["language_code"].value_counts(dropna=False)
language_value_counts


A lot of languages are unique or very rare, we might want to group them together for modelling.


In [ ]:
# Plot mean average_rating by language_code with 95% confidence intervals.

fig, ax = plt.subplots(figsize=(9, 5))

language_value_counts = df_model["language_code"].value_counts(dropna=False)

language_rating_plot_df = df_model[["language_code", "average_rating"]].copy()

language_rating_order = (
    language_rating_plot_df.groupby("language_code", dropna=False)["average_rating"]
    .mean()
    .sort_values(ascending=False)
    .index
)

sns.pointplot(
    data=language_rating_plot_df,
    x="language_code",
    y="average_rating",
    order=language_rating_order,
    errorbar=("ci", 95),
    color="#2a9d8f",
    ax=ax,
)
ax.set_title("Language & ratings")
ax.set_xlabel("Language Code")
ax.set_ylabel("Mean average rating")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# check for multi-language books
multi_language_books = (
    df_model.groupby(["title", "authors", "average_rating"])["language_code"]
    .agg(n_languages="nunique", languages=lambda language_codes: sorted(language_codes.dropna().unique()))
    .reset_index()
    .query("n_languages > 1")
    .sort_values("n_languages", ascending=False)
    .head(50)
)
multi_language_books


In [ ]:
# average rating for multi-language books
round(multi_language_books["average_rating"].mean(), 2)


The average rating for books with multiple languages is about the same as the average rating for the dataset -> not a strong indicator of the quality of the book. We might want to test the models without this feature first and add it as a sensitivity to see if it improves the model.


**[Decision]** Create a broader English / non-English feature for modelling.


In [ ]:
# Create a binary English-language feature for modelling.
ENGLISH_LANGUAGE_CODES = {"eng", "en-US", "en-GB", "en-CA", "enm"}

df_model["is_english"] = df_model["language_code"].isin(ENGLISH_LANGUAGE_CODES).astype(int)

english_language_summary = (
    df_model.groupby("is_english")
    .agg(
        books=("average_rating", "size"),
        mean_rating=("average_rating", "mean"),
        median_rating=("average_rating", "median"),
    )
    .reset_index()
)
english_language_summary["share_of_df_model"] = english_language_summary["books"] / len(df_model)
print("English codes mapped to is_english = 1:", sorted(ENGLISH_LANGUAGE_CODES))

english_language_summary


English-language books clearly dominate the dataset, so a single binary `is_english` feature is enough for a first modelling pass without keeping a sparse language grouping feature. Also, the average rating for different English languages is not significantly different so it is reasonable to group them together.


### 2.8 `authors`


Visual inspection shows that some books have a lot of authors, separated by a "/". We might want to separate the strings to identify each author.


#### First look


In [ ]:
# check what is the maximum number of authors for one book
author_count_per_book = df_model["authors"].str.split("/").str.len()
print("max number of authors for one book:", author_count_per_book.max())


51 authors for one book !


In [ ]:
author_preview_columns = ["title", "authors"]
df_model.loc[author_count_per_book == author_count_per_book.max(), author_preview_columns]


**[Decision]** To avoid empty data, and because the first author is usually the main one (confirmed below), let's add a column for the first author only, and one column with the number of authors


In [ ]:
# creating new columns
df_model["first_author"] = df_model["authors"].str.split(
    "/").str[0].str.strip()
df_model["n_authors"] = author_count_per_book


In [ ]:
# Plot the distribution of author counts to show how many authors most books have.
plt.figure(figsize=(10, 6))
sns.histplot(df_model["n_authors"], bins=range(1, author_count_per_book.max() + 2))
plt.title("Distribution of Number of Authors per Book")
plt.xlabel("Number of Authors")
plt.ylabel("Count")
plt.show()


It looks like books with more than 10 authors are rare


In [ ]:
# Print number of unique authors vs first author
print(f"Number of unique authors : {df_model['authors'].nunique()}")
print(f"Number of unique first authors : {df_model['first_author'].nunique()}")


In [ ]:
# Compare the distribution of author counts and how ratings vary with the number of authors.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.countplot(
    data=df_model.assign(author_count_bucket=df_model["n_authors"].clip(upper=10).astype(int)),
    x="author_count_bucket",
    color="#2a9d8f",
    ax=axes[0],
)
axes[0].set_title("Authors per book (capped at 10 for readability)")
axes[0].set_xlabel("Number of authors")
axes[0].set_ylabel("Books")

high_author_count_share = (df_model["n_authors"] > 10).mean()
axes[0].text(
    0.98,
    0.95,
    f"{high_author_count_share:.2%} of rows have >10 authors",
    transform=axes[0].transAxes,
    ha="right",
    va="top",
    fontsize=9,
)

sns.boxplot(data=df_model, x="n_authors", y="average_rating",
            color="#e9c46a", ax=axes[1])
axes[1].set_title("Rating vs author count (raw counts)")
axes[1].set_xlabel("Number of authors")
plt.tight_layout()
plt.show()


No clear correlation between number of authors and average rating.


#### Author frequency features


One simple way to keep some author information without one-hot encoding thousands of author names is to add `first_author_frequency`: how often the first author appears in `df_model`. This is not based on the target rating, only on row counts.

For this EDA section, the frequency is computed on `df_model`. In the final modelling pipeline, this mapping should be learned from the training set only, then applied to validation/test/external data. A separate `is_top_first_author` flag is not needed if we keep the actual frequency value.


In [ ]:
# Frequency encoding candidate for author information.
# Keep only one frequency feature: how many rows share the same first_author.
first_author_frequency_counts = df_model["first_author"].value_counts()

df_model["first_author_frequency"] = (
    df_model["first_author"].map(first_author_frequency_counts).fillna(0).astype(int)
)

author_feature_preview_columns = [
    "first_author",
    "authors",
    "n_authors",
    "first_author_frequency",
]

df_model[author_feature_preview_columns].head()


In [ ]:
# Spearman correlation with average_rating for the selected numeric author features.
author_numeric_features = [
    "n_authors",
    "first_author_frequency",
]

author_feature_correlation = (
    df_model[author_numeric_features + ["average_rating"]]
    .corr(method="spearman")
    .loc[author_numeric_features, "average_rating"]
    .reset_index()
    .rename(columns={"index": "feature", "average_rating": "spearman_corr"})
    .assign(abs_spearman=lambda data: data["spearman_corr"].abs())
    .sort_values("abs_spearman", ascending=False)
    .drop(columns="abs_spearman")
)

author_feature_correlation.round(4)


`n_authors` has only a small positive rank correlation with `average_rating`, and `first_author_frequency` is weaker. This means author frequency may help slightly in combination with other features, but it should not be expected to explain ratings by itself.


### 2.9 `title`


#### Duplicate checks


We already established that there are no strict duplicates, since bookID, isbn and isbn13 are all unique


#### Title duplicates


In [ ]:
# number of duplicated titles
print("number of duplicated titles :",(df_model["title"].value_counts() > 1).sum())


There are a few duplicated titles, but they might have different authors


In [ ]:
#  duplicated titles & authors
duplicate_title_author_counts = df_model[["title", "authors"]].value_counts()
print("number of duplicated titles & authors :",(duplicate_title_author_counts > 1).sum())
# (df_model[["title","authors"]].value_counts()>1).sum()


In [ ]:
# duplicated titles & first author
duplicate_title_first_author_counts = df_model[["title", "first_author"]].value_counts()
print("number of duplicated titles & first author :",(duplicate_title_first_author_counts > 1).sum())


Here we see that the first author check is more reliable than the authors check, closer to the number of duplicated titles. What could be interesting is to check duplicated titles with different first author


In [ ]:
# Show titles with the same title but different first_author
title_first_author_counts = df_model.groupby("title")["first_author"].nunique()
duplicate_titles_with_different_first_author = title_first_author_counts[title_first_author_counts > 1].index
df_model[df_model["title"].isin(duplicate_titles_with_different_first_author)][["title", "first_author", "average_rating","num_pages","language_code"]]


In [ ]:
# check if there exist books with same title and authors, but different ratings
print("number of books with same title and authors, but different ratings :", (df_model.groupby(["title","first_author"])["average_rating"].nunique() > 1).sum())


Good news, duplicated books (by title & authors) all have the same ratings


In [ ]:
# number of books with same title but different authors
same_title_different_authors_mask = df_model.groupby(
    ["title"])["authors"].nunique() > 1
print("number of books with same title but different authors :",same_title_different_authors_mask.sum())


In [ ]:
df_model["title"].value_counts()


In [ ]:
df_model[df_model["title"] == "the iliad"][:]


From this example, we can infer that even though the authors are differents, the first author is the same.
We might use this information to merge the books with the same title but different authors.


In [ ]:
# number for books with same title but different first author
same_title_different_first_author_mask = df_model.groupby(
    ["title"])["first_author"].nunique() > 1
print("number of books with same title but different first author :",same_title_different_first_author_mask.sum())


In [ ]:
print("number of books with same title and first_author but different average rating :",
      (df_model.groupby(["title", "first_author"])
       ["average_rating"].nunique() > 1).sum()
)


In [ ]:
# Generate and display the books with the same title and first_author but different average_rating, showing only simple aggregated fields and lists of values
books_with_rating_variants = (
    df_model.groupby(["title", "first_author"])
    .agg(
        n_unique_average_rating=("average_rating", "nunique"),
        ratings_list=("average_rating", lambda ratings: list(ratings.unique())),
        num_pages_list=("num_pages", lambda page_counts: list(page_counts.unique())),
        language_code_list=("language_code", lambda language_codes: list(language_codes.unique())),
        publisher_list=("publisher", lambda publishers: list(publishers.unique())),
        publication_year_list=("publication_year", lambda publication_years: list(publication_years.unique())),
    )
    .reset_index()
    .query("n_unique_average_rating > 1")
)
books_with_rating_variants


for these 5 books, it turns out that title + first author is not a good identifier for duplicates, as they have other differentiating fields. We can leave as is.


As seen earlier, there are no truly duplicated rows, but some of the rows only differ by publisher or language code : average_rating is the same when title & authors are the same (and mostly the same when using first_author).


[Decision] For now, let's keep these rows as they are. The duplicated titles might simply be different editions, publishers, languages or collections.


#### Collections and series


In [ ]:
# Add title-based flags for collections and series entries.
# The identifiers for series are "#[number]" / "Volume [number]" / "Vol. [number]" / "Books [number]" / "Part [number]" / "Tome [number]".
TITLE_REGEX_FLAGS = re.IGNORECASE | re.VERBOSE

SERIES_NUMBER_PATTERN = r"(?:\d+(?:\.\d+)?|[ivxlcdm]+|one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve)"
SERIES_NUMBER_RANGE_PATTERN = rf"{SERIES_NUMBER_PATTERN}(?:\s*(?:[-\u2013\u2014,/&]|and)\s*{SERIES_NUMBER_PATTERN})*"

collection_title_pattern = r"""
\b(?:box(?:ed)?\s+set|omnibus|antholog(?:y|ies)|collection)\b
|
\b(?:complete|collected|selected|essential|major)\s+
(?:works|novels|stories|short\s+stories|poems|poetry|plays|writings|letters|essays|tragedies|adventures|memoirs)\b
|
\b\d+\s*(?:book|volume)s?\s+(?:box(?:ed)?\s+set|set|collection)\b
|
\b(?:\d+\s+)?volumes?\s+set\b
"""

series_title_pattern = rf"""
\([^)]*(?:\#\s*{SERIES_NUMBER_RANGE_PATTERN}|\b(?:vol(?:ume)?s?\.?|books?|part|tome)\s+{SERIES_NUMBER_RANGE_PATTERN})[^)]*\)
|
(?<!\w)\#\s*{SERIES_NUMBER_RANGE_PATTERN}\b
|
\b(?:vol(?:ume)?s?\.?|part|tome)\s+{SERIES_NUMBER_RANGE_PATTERN}\b
"""

title_series = df_model["title"].fillna("")

df_model["is_collection"] = title_series.str.contains(
    collection_title_pattern, regex=True, flags=TITLE_REGEX_FLAGS, na=False)
df_model["is_series"] = title_series.str.contains(
    series_title_pattern, regex=True, flags=TITLE_REGEX_FLAGS, na=False)


In [ ]:
# Compare the rating level of books flagged as collections or series entries.
# The comparison uses both mean and median because average_rating is bounded and slightly skewed.
collection_series_summary_rows = []

for label, flag_column in {
    "Collections": "is_collection",
    "Series entries": "is_series",
}.items():
    flagged_books = df_model[df_model[flag_column]]
    unflagged_books = df_model[~df_model[flag_column]]

    collection_series_summary_rows.append({
        "category": label,
        "books": len(flagged_books),
        "share_of_df_model": len(flagged_books) / len(df_model),
        "mean_rating": flagged_books["average_rating"].mean(),
        "median_rating": flagged_books["average_rating"].median(),
        "non_flagged_mean_rating": unflagged_books["average_rating"].mean(),
        "mean_difference": flagged_books["average_rating"].mean() - unflagged_books["average_rating"].mean(),
    })

collection_series_summary_df = pd.DataFrame(collection_series_summary_rows)
collection_series_summary_df.round(3)


In [ ]:
# Plot mean and median ratings side by side to check whether the difference is driven by outliers.
collection_series_plot_df = collection_series_summary_df.melt(
    id_vars="category",
    value_vars=["mean_rating", "median_rating"],
    var_name="metric",
    value_name="rating",
)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=collection_series_plot_df, x="category", y="rating", hue="metric", ax=ax)
ax.set_title("Average and Median Rating for Collections and Series Entries")
ax.set_xlabel("")
ax.set_ylabel("Rating")
ax.set_ylim(3.7, 4.35)
ax.legend(title="")

for bar_container in ax.containers:
    ax.bar_label(bar_container, fmt="%.2f")

plt.tight_layout()
plt.show()


Collections represent a small share of `df_model`, so their higher mean and median rating should be interpreted carefully. The pattern is still plausible: collections, complete works, boxed sets, and selected works are often created for books or authors with an existing audience.

Series entries are much more common. Their mean rating is only slightly higher than non-series books, so `is_series` may add some signal, but it is unlikely to be a strong standalone predictor. Both flags should be tested during modelling rather than treated as automatically useful.


### 2.10 `publisher`


#### First look

Publisher was already used earlier in the audiobook processing: `is_audio` was derived from title and publisher text because several low or zero page-count records looked like audiobook publishers.


In [ ]:
publisher_value_counts = df_model["publisher"].value_counts(dropna=False)
publisher_value_counts.head(20)


In [ ]:
publisher_overview = pd.DataFrame({
    "metric": [
        "unique_publishers",
        "single_book_publishers",
        "publishers_with_fewer_than_5_books",
        "top_10_publishers_share",
    ],
    "value": [
        df_model["publisher"].nunique(dropna=False),
        int((publisher_value_counts == 1).sum()),
        int((publisher_value_counts < 5).sum()),
        publisher_value_counts.head(10).sum() / len(df_model),
    ],
})

publisher_overview


In [ ]:
TOP_PUBLISHER_PLOT_COUNT = 15

top_publisher_count_df = (
    publisher_value_counts
    .head(TOP_PUBLISHER_PLOT_COUNT)
    .rename_axis("publisher")
    .reset_index(name="books")
)

plt.figure(figsize=(9, 6))
sns.barplot(
    data=top_publisher_count_df,
    y="publisher",
    x="books",
    color="#2a9d8f",
)
plt.title("Most frequent publishers")
plt.xlabel("Number of books")
plt.ylabel("Publisher")
plt.tight_layout()
plt.show()


The publisher column has many rare values, so one-hot encoding all publishers would create a sparse feature set. A compact summary of the most common publishers is still useful to check whether some of them are mostly audiobooks, collections, or series entries.


In [ ]:
TOP_PUBLISHER_COUNT = 15

top_publisher_names = publisher_value_counts.head(TOP_PUBLISHER_COUNT).index

publisher_summary = (
    df_model[df_model["publisher"].isin(top_publisher_names)]
    .groupby("publisher")
    .agg(
        books=("average_rating", "size"),
        mean_rating=("average_rating", "mean"),
        median_rating=("average_rating", "median"),
        rating_std=("average_rating", "std"),
        audiobook_share=("is_audio", "mean"),
        collection_share=("is_collection", "mean"),
        series_share=("is_series", "mean"),
    )
    .sort_values("books", ascending=False)
    .reset_index()
)
publisher_summary["share_of_df_model"] = publisher_summary["books"] / len(df_model)

publisher_summary.round(3)


#### Publisher frequency feature

As with `first_author_frequency`, a simple publisher frequency encoding keeps broad publisher information without creating thousands of dummy columns. This is not based on the target rating; it only counts how often each publisher appears in `df_model`.


In [ ]:
publisher_frequency_counts = df_model["publisher"].value_counts()

df_model["publisher_frequency"] = (
    df_model["publisher"].map(publisher_frequency_counts).fillna(0).astype(int)
)

publisher_feature_preview_columns = [
    "publisher",
    "publisher_frequency",
    "is_audio",
    "is_collection",
    "is_series",
    "average_rating",
]

df_model[publisher_feature_preview_columns].head()


In [ ]:
publisher_numeric_features = ["publisher_frequency"]

publisher_feature_correlation = (
    df_model[publisher_numeric_features + ["average_rating"]]
    .corr(method="spearman")
    .loc[publisher_numeric_features, "average_rating"]
    .reset_index()
    .rename(columns={"index": "feature", "average_rating": "spearman_corr"})
)

publisher_feature_correlation.round(4)


`publisher_frequency` is a simple candidate feature for the first modelling pass. As the spearman correlation shows, there don't seeme to be a strong relationship between publisher frequency and average rating.


------------
## 3. Modelling


------------
### 3.1 Additional feature engineering for modelling

Before model training, a few additional reusable features are created from the cleaned `df_model` table. These features are predictor-side only; no target-derived rating group is used as an input.


#### Create Modelling Features
This block adds time, log-count, language grouping, review-ratio, and title-structure features used by the model families below.


In [ ]:
# This cell creates modelling-only features from the cleaned `df_model` table.
# The new columns are simple transformations that can be reused by the three model families.
REFERENCE_YEAR = 2026
MIN_LANGUAGE_COUNT = 50

df_model = df_model.copy()

# Time since publication, using a fixed reference year for reproducibility.
df_model["book_age"] = (REFERENCE_YEAR - df_model["publication_year"]).clip(lower=0)
df_model["publication_decade"] = ((df_model["publication_year"].astype(int) // 10) * 10).astype(str) + "s"

# Log transforms reduce the influence of very large count values.
df_model["log_ratings_count"] = np.log1p(df_model["ratings_count"])
df_model["log_text_reviews_count"] = np.log1p(df_model["text_reviews_count"])
df_model["log_num_pages"] = np.log1p(df_model["num_pages"])
df_model["log_first_author_frequency"] = np.log1p(df_model["first_author_frequency"])
df_model["log_publisher_frequency"] = np.log1p(df_model["publisher_frequency"])

# Written reviews as a share of all ratings.
df_model["review_to_rating_ratio"] = (df_model["text_reviews_count"] / df_model["ratings_count"].replace(0, np.nan)).fillna(0)
df_model["log_review_share"] = np.log1p(df_model["review_to_rating_ratio"])

# Keep common languages and group rare ones.
language_for_grouping = df_model["language_code"].fillna("missing_language")
language_counts = language_for_grouping.value_counts()
common_languages = language_counts[language_counts >= MIN_LANGUAGE_COUNT].index

df_model["language_code_grouped"] = language_for_grouping.where(language_for_grouping.isin(common_languages),"other_language",)

# Simple title structure features.
title_text = df_model["title"].fillna("")
df_model["title_word_count"] = title_text.str.split().str.len().fillna(0).astype(int)
df_model["title_char_count"] = title_text.str.len().fillna(0).astype(int)
df_model["title_has_subtitle"] = title_text.str.contains(r":", regex=True, na=False).astype(int)
df_model["title_has_parentheses"] = title_text.str.contains(r"[()]", regex=True, na=False).astype(int)

modelling_engineered_features = [
    "book_age",
    "publication_decade",
    "log_ratings_count",
    "log_text_reviews_count",
    "log_num_pages",
    "log_first_author_frequency",
    "log_publisher_frequency",
    "review_to_rating_ratio",
    "log_review_share",
    "language_code_grouped",
    "title_word_count",
    "title_char_count",
    "title_has_subtitle",
    "title_has_parentheses",
]

feature_missing_check = (
    df_model[modelling_engineered_features]
    .isna()
    .sum()
    .rename("missing_values")
    .reset_index()
    .rename(columns={"index": "feature"})
)

feature_missing_check


### 3.2 Feature selection

The target is `average_rating`, so this is a regression problem. I use feature sets adapted to each model instead of forcing every model to use exactly the same input columns.

- `LinearRegression` uses log-transformed numeric features, simple flags, grouped language, publication decade, and the existing band features with one-hot encoding.
- `HistGradientBoostingRegressor` uses numeric and log-numeric features only. This keeps the sklearn boosted-tree model simple and avoids sparse categorical inputs.
- `CatBoostRegressor` uses numeric and log-numeric features plus native categorical columns such as first author, publisher, grouped language, publication decade, and the band columns.

I still exclude identifiers such as `isbn` and `isbn13`, and I do not use any target-derived rating group as a predictor.


#### Modelling Imports And Constants
This block imports only the modelling tools used below and defines shared constants for the target, split, random seed, and CSV log path.


In [ ]:
# This cell imports the modelling tools and defines which features each model will receive.
# The feature sets are adapted to the strengths of linear regression, sklearn boosting, and CatBoost.
import uuid
from datetime import datetime
from itertools import product
from pathlib import Path

from catboost import CatBoostRegressor
from IPython.display import display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Shared modelling constants.
TARGET = "average_rating"
RANDOM_STATE = 42
TEST_SIZE = 0.20
RATING_BAND_COUNT = 10
MODEL_LOG_PATH = Path("../logs/model_runs.csv")
RUN_MANUAL_TUNING = False


#### Linear Regression Features
Linear regression gets scaled numeric inputs plus a compact set of one-hot categorical and band features.


In [ ]:
# Linear regression uses scaled numeric features and one-hot encoded categorical/band features.
LINEAR_NUMERIC_FEATURES = [
    "book_age",
    "log_ratings_count",
    "log_text_reviews_count",
    "log_num_pages",
    "review_to_rating_ratio",
    "log_review_share",
    "n_authors",
    "log_first_author_frequency",
    "log_publisher_frequency",
    "title_word_count",
    "title_char_count",
    "title_has_subtitle",
    "title_has_parentheses",
    "num_pages_missing",
    "is_audio",
    "is_collection",
    "is_series",
]

LINEAR_CATEGORICAL_FEATURES = [
    "language_code_grouped",
    "publication_decade",
    "ratings_count_band",
    "page_count_band",
    "text_review_count_ntile",
]


#### Boosted-Tree Features
The boosted-tree models use the same numeric base. CatBoost additionally receives native categorical metadata.


In [ ]:
# HistGradientBoosting uses numeric columns only, including raw and log-transformed counts.
TREE_NUMERIC_FEATURES = [
    "ratings_count",
    "text_reviews_count",
    "num_pages",
    "book_age",
    "log_ratings_count",
    "log_text_reviews_count",
    "log_num_pages",
    "review_to_rating_ratio",
    "log_review_share",
    "n_authors",
    "first_author_frequency",
    "log_first_author_frequency",
    "publisher_frequency",
    "log_publisher_frequency",
    "title_word_count",
    "title_char_count",
    "title_has_subtitle",
    "title_has_parentheses",
    "num_pages_missing",
    "is_audio",
    "is_collection",
    "is_series",
]

# CatBoost can use categorical book metadata directly.
CATBOOST_CATEGORICAL_FEATURES = [
    "first_author",
    "publisher",
    "language_code_grouped",
    "publication_decade",
    "ratings_count_band",
    "page_count_band",
    "text_review_count_ntile",
]


#### Model Feature Registry
This registry is the single lookup used later to select the right feature set for each model family.


In [ ]:
# Central dictionary used later to choose the correct features for each model.
MODEL_FEATURES = {
    "Linear regression": {
        "numeric_features": LINEAR_NUMERIC_FEATURES,
        "categorical_features": LINEAR_CATEGORICAL_FEATURES,
    },
    "HistGradientBoosting": {
        "numeric_features": TREE_NUMERIC_FEATURES,
        "categorical_features": [],
    },
    "CatBoost": {
        "numeric_features": TREE_NUMERIC_FEATURES,
        "categorical_features": CATBOOST_CATEGORICAL_FEATURES,
    },
}

# Display a quick count of the inputs passed to each model.
model_feature_overview = pd.DataFrame([
    {
        "model": model_name,
        "numeric_features": len(feature_config["numeric_features"]),
        "categorical_features": len(feature_config["categorical_features"]),
        "total_input_columns": (
            len(feature_config["numeric_features"])
            + len(feature_config["categorical_features"])
        ),
    }
    for model_name, feature_config in MODEL_FEATURES.items()
])

model_feature_overview


### 3.3 Train/test split

The same split is used for all models. Since `average_rating` is concentrated in a narrow range, the split is stratified by quantile bands.


#### Stratified Train/test Split
The helper is defined directly before it is used to create the fixed modelling split and inspect basic target statistics.


In [ ]:
# Split helper and split summary.
# This is kept separate from model fitting so the train/test check is easy to inspect.
def create_stratified_rating_split(data):
    rating_band_intervals = pd.qcut(
        data[TARGET],
        q=RATING_BAND_COUNT,
        duplicates="drop",
    )
    rating_band_codes = rating_band_intervals.cat.codes

    train_index, test_index = train_test_split(
        data.index,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=rating_band_codes,
    )

    return train_index, test_index, rating_band_intervals


MODEL_TRAIN_INDEX, MODEL_TEST_INDEX, model_rating_bands = create_stratified_rating_split(df_model)

split_check_df = pd.concat([
    pd.DataFrame({
        "split": "train",
        TARGET: df_model.loc[MODEL_TRAIN_INDEX, TARGET],
        "rating_band": model_rating_bands.loc[MODEL_TRAIN_INDEX].astype(str),
    }),
    pd.DataFrame({
        "split": "test",
        TARGET: df_model.loc[MODEL_TEST_INDEX, TARGET],
        "rating_band": model_rating_bands.loc[MODEL_TEST_INDEX].astype(str),
    }),
])

split_summary = (
    split_check_df
    .groupby("split")
    .agg(
        books=(TARGET, "size"),
        mean_rating=(TARGET, "mean"),
        median_rating=(TARGET, "median"),
        min_rating=(TARGET, "min"),
        max_rating=(TARGET, "max"),
        rating_std=(TARGET, "std"),
    )
    .reindex(["train", "test"])
)
split_summary["share_of_data"] = split_summary["books"] / len(df_model)

split_summary.round(3)


#### Rating-Band Balance Check
This confirms that the stratified split kept a similar target-band distribution in train and test.


In [ ]:
# Check whether each rating band has a similar share in train and test.
rating_band_split_share = pd.crosstab(
    split_check_df["rating_band"],
    split_check_df["split"],
    normalize="columns",
).reindex(columns=["train", "test"])

(rating_band_split_share * 100).round(1)


### 3.4 Active model training and CSV logging

The normal notebook run trains only the active configurations. Manual tuning remains optional in the next section.


#### Active Model Configurations
These are the configurations used for regular runs. Once tuning is done, copy the selected parameters here and skip the tuning grid.


In [ ]:
# Active model configurations used for normal notebook runs.
# After running the tuning grid, copy the best parameters here and skip Section 3.5 on later runs.
active_model_configs = [
    {
        "model": "Linear regression",
        "estimator": LinearRegression(),
        "hyperparameters": {},
    },
    {
        "model": "HistGradientBoosting",
        "estimator": HistGradientBoostingRegressor(
            max_iter=200,
            learning_rate=0.03,
            max_leaf_nodes=63,
            l2_regularization=0.05,
            random_state=RANDOM_STATE,
        ),
        "hyperparameters": {
            "max_iter": 200,
            "learning_rate": 0.03,
            "max_leaf_nodes": 63,
            "l2_regularization": 0.05,
        },
    },
    {
        "model": "CatBoost",
        "estimator": CatBoostRegressor(
            iterations=1500,
            learning_rate=0.06,
            depth=5,
            l2_leaf_reg=5,
            loss_function="RMSE",
            eval_metric="MAE",
            random_seed=RANDOM_STATE,
            verbose=False,
            allow_writing_files=False,
        ),
        "hyperparameters": {
            "iterations": 1500,
            "learning_rate": 0.06,
            "depth": 5,
            "l2_leaf_reg": 5,
        },
    },
]


#### Sklearn Preprocessing Pipeline
These helpers are placed immediately before model fitting because they are used by `fit_and_evaluate_model` for the sklearn models.


In [ ]:
# Preprocessing helpers used later by the sklearn models.
# CatBoost does not use this pipeline because it can receive categorical columns directly.
def make_one_hot_encoder():
    """Create a OneHotEncoder that works across recent sklearn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def prepare_feature_matrix(data, numeric_features, categorical_features):
    """Select the columns needed by one model family and clean categorical values."""
    feature_names = numeric_features + categorical_features
    X = data[feature_names].copy()

    bool_columns = X.select_dtypes(include="bool").columns
    X[bool_columns] = X[bool_columns].astype(int)

    for feature in categorical_features:
        X[feature] = X[feature].astype("object").where(X[feature].notna(), "Missing").astype(str)

    return X


def build_sklearn_pipeline(model_name, model, numeric_features, categorical_features):
    """Build preprocessing plus estimator for linear regression and sklearn boosting."""
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]

    if model_name == "Linear regression":
        numeric_steps.append(("scaler", StandardScaler()))

    transformers = [
        ("num", Pipeline(steps=numeric_steps), numeric_features),
    ]

    if categorical_features:
        categorical_transformer = Pipeline(steps=[
            ("onehot", make_one_hot_encoder()),
        ])
        transformers.append(("cat", categorical_transformer, categorical_features))

    return Pipeline(steps=[
        ("preprocessor", ColumnTransformer(transformers=transformers)),
        ("model", clone(model)),
    ])


#### Fit And Score Configurations
The scoring helpers are defined here because they are used immediately to fit every active model and build the result table.


In [ ]:
# Formatting and scoring helpers are placed here because this is where model results are created.
def format_hyperparameters(hyperparameters):
    if not hyperparameters:
        return "initial"
    return ", ".join(f"{key}={value}" for key, value in hyperparameters.items())


def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }


def fit_and_evaluate_model(config, train_index, test_index):
    """Fit one model configuration and return metrics, fitted model, and log metadata."""
    model_name = config["model"]
    feature_config = MODEL_FEATURES[model_name]
    numeric_features = feature_config["numeric_features"]
    categorical_features = feature_config["categorical_features"]

    X = prepare_feature_matrix(df_model, numeric_features, categorical_features)
    y = df_model[TARGET]

    X_train = X.loc[train_index]
    X_test = X.loc[test_index]
    y_train = y.loc[train_index]
    y_test = y.loc[test_index]

    if model_name == "CatBoost":
        fitted_model = clone(config["estimator"])
        fitted_model.fit(
            X_train,
            y_train,
            cat_features=categorical_features,
        )
    else:
        fitted_model = build_sklearn_pipeline(
            model_name,
            config["estimator"],
            numeric_features,
            categorical_features,
        )
        fitted_model.fit(X_train, y_train)

    y_pred = fitted_model.predict(X_test)
    metrics = regression_metrics(y_test, y_pred)
    hyperparameter_label = format_hyperparameters(config.get("hyperparameters", {}))

    result_row = {
        "model": model_name,
        "n_numeric_features": len(numeric_features),
        "n_categorical_features": len(categorical_features),
        "hyperparameters": hyperparameter_label,
    }
    result_row.update(metrics)

    log_row = {
        "model": model_name,
        "model_type": type(config["estimator"]).__name__,
        "target": TARGET,
        "random_state": RANDOM_STATE,
        "test_size": TEST_SIZE,
        "rating_band_count": RATING_BAND_COUNT,
        "n_train_rows": len(train_index),
        "n_test_rows": len(test_index),
        "numeric_features": list(numeric_features),
        "categorical_features": list(categorical_features),
        "hyperparameters": config.get("hyperparameters", {}),
        "metrics": {metric: float(value) for metric, value in metrics.items()},
    }

    fitted_key = (model_name, hyperparameter_label)
    return result_row, fitted_key, fitted_model, log_row


def evaluate_model_configs(configs, train_index, test_index):
    """Evaluate several configurations and keep the records needed for logging."""
    rows = []
    fitted_models = {}
    log_rows = []

    for config in configs:
        row, fitted_key, fitted_model, log_row = fit_and_evaluate_model(config, train_index, test_index)
        rows.append(row)
        fitted_models[fitted_key] = fitted_model
        log_rows.append(log_row)

    return pd.DataFrame(rows), fitted_models, log_rows


model_results, fitted_models, model_log_rows = evaluate_model_configs(
    active_model_configs,
    MODEL_TRAIN_INDEX,
    MODEL_TEST_INDEX,
)

model_results.sort_values("MAE").round({"MAE": 3, "RMSE": 3, "R2": 3})


#### Flatten Model Runs For CSV
These helpers convert nested feature lists, hyperparameters, and metric dictionaries into one flat row per model run.


In [ ]:
# Flatten feature lists and hyperparameters before writing model runs to CSV.
def all_model_feature_names():
    """Return every feature that can appear in the model feature configurations."""
    return sorted({
        feature
        for feature_config in MODEL_FEATURES.values()
        for feature in (
            feature_config["numeric_features"]
            + feature_config["categorical_features"]
        )
    })


def normalize_hyperparameters(hyperparameters):
    """Map equivalent hyperparameter names from different model APIs to shared log columns."""
    parameter_aliases = {
        "max_iter": "n_estimators",
        "iterations": "n_estimators",
        "depth": "tree_depth",
        "l2_leaf_reg": "l2_regularization",
    }

    normalized_parameters = {}
    for parameter_name, parameter_value in hyperparameters.items():
        normalized_name = parameter_aliases.get(parameter_name, parameter_name)
        normalized_parameters[normalized_name] = parameter_value

    return normalized_parameters


def flatten_model_log_row(log_row):
    """Convert one nested model log row into a flat CSV-friendly dictionary."""
    numeric_features = set(log_row["numeric_features"])
    categorical_features = set(log_row["categorical_features"])
    selected_features = numeric_features | categorical_features

    flattened_row = {
        "run_id": str(uuid.uuid4()),
        "datetime": datetime.now().astimezone().isoformat(timespec="seconds"),
        "model": log_row["model"],
        "model_type": log_row["model_type"],
        "target": log_row["target"],
        "random_state": log_row["random_state"],
        "test_size": log_row["test_size"],
        "rating_band_count": log_row["rating_band_count"],
        "n_train_rows": log_row["n_train_rows"],
        "n_test_rows": log_row["n_test_rows"],
        "n_features": len(selected_features),
        "n_numeric_features": len(numeric_features),
        "n_categorical_features": len(categorical_features),
    }
    flattened_row.update(log_row["metrics"])

    for parameter_name, parameter_value in normalize_hyperparameters(log_row["hyperparameters"]).items():
        flattened_row[f"param_{parameter_name}"] = parameter_value

    for feature_name in all_model_feature_names():
        flattened_row[f"feature_{feature_name}"] = int(feature_name in selected_features)

    return flattened_row


#### Append Runs To The CSV Log
The logger appends new runs while preserving old rows and adding new columns if later feature experiments introduce new inputs.


In [ ]:
# Append flattened model runs to the CSV log with a stable column order.
def ordered_model_log_columns(log_rows):
    """Keep metadata first, then metrics, hyperparameters, and one-hot feature columns."""
    base_columns = [
        "run_id",
        "datetime",
        "model",
        "model_type",
        "target",
        "random_state",
        "test_size",
        "rating_band_count",
        "n_train_rows",
        "n_test_rows",
        "n_features",
        "n_numeric_features",
        "n_categorical_features",
    ]
    metric_columns = ["MAE", "RMSE", "R2"]
    parameter_columns = sorted({
        column
        for row in log_rows
        for column in row
        if column.startswith("param_")
    })
    feature_columns = sorted({
        column
        for row in log_rows
        for column in row
        if column.startswith("feature_")
    })
    known_columns = set(base_columns + metric_columns + parameter_columns + feature_columns)
    other_columns = sorted({
        column
        for row in log_rows
        for column in row
        if column not in known_columns
    })

    return base_columns + metric_columns + parameter_columns + feature_columns + other_columns


def log_model_runs(log_rows):
    if not log_rows:
        return

    MODEL_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

    new_log_df = pd.DataFrame([flatten_model_log_row(log_row) for log_row in log_rows])

    if MODEL_LOG_PATH.exists():
        existing_log_df = pd.read_csv(MODEL_LOG_PATH)
        combined_log_df = pd.concat([existing_log_df, new_log_df], ignore_index=True, sort=False)
    else:
        combined_log_df = new_log_df

    ordered_columns = ordered_model_log_columns(combined_log_df.to_dict("records"))
    combined_log_df = combined_log_df.reindex(columns=ordered_columns)

    feature_columns = [column for column in ordered_columns if column.startswith("feature_")]
    combined_log_df[feature_columns] = combined_log_df[feature_columns].fillna(0).astype(int)

    combined_log_df.to_csv(MODEL_LOG_PATH, index=False)


log_model_runs(model_log_rows)


### 3.5 Manual tuning

This section is optional after the first tuning run. It searches for better parameters for `HistGradientBoostingRegressor` and `CatBoostRegressor`.

The regular notebook run uses `active_model_configs` in Section 3.4. After the grid finds better parameters, copy the best values into `active_model_configs` and then skip the tuning cells below to avoid rerunning the full grid every time.

Set `RUN_MANUAL_TUNING = True` in the modelling constants only when you want to rerun the full grid. The current default is `False`, so these cells are skipped during normal runs.


#### Tuning Grid Definition
The grids focus around the best ranges seen so far and include a compact count check before fitting anything.


In [ ]:
# Manual tuning is guarded so normal notebook runs do not launch the full grid.
if RUN_MANUAL_TUNING:
    # The values focus around plausible ranges and the best settings found in the smaller first pass.
    hist_gradient_param_grid = {
        "max_iter": [200, 300, 400, 600],
        "learning_rate": [0.02, 0.03, 0.05],
        "max_leaf_nodes": [15, 31, 63],
        "l2_regularization": [0.01, 0.05, 0.1],
    }

    catboost_param_grid = {
        "iterations": [900, 1200, 1500],
        "learning_rate": [0.04, 0.06, 0.08],
        "depth": [5, 6, 7, 8],
        "l2_leaf_reg": [3, 5, 7, 9],
    }

    def expand_param_grid(param_grid):
        """Return every parameter combination from a grid dictionary."""
        parameter_names = list(param_grid.keys())
        parameter_values = [param_grid[name] for name in parameter_names]

        return [
            dict(zip(parameter_names, values))
            for values in product(*parameter_values)
        ]

    hist_gradient_tuning_configs = expand_param_grid(hist_gradient_param_grid)
    catboost_tuning_configs = expand_param_grid(catboost_param_grid)

    # Check the size of the grid before fitting the models.
    tuning_grid_summary = pd.DataFrame([
        {
            "model": "HistGradientBoosting",
            "grid_configurations": len(hist_gradient_tuning_configs),
            "status": "ready",
        },
        {
            "model": "CatBoost",
            "grid_configurations": len(catboost_tuning_configs),
            "status": "ready",
        },
    ])
    tuning_grid_summary = pd.concat([
        tuning_grid_summary,
        pd.DataFrame([{
            "model": "Total",
            "grid_configurations": tuning_grid_summary["grid_configurations"].sum(),
            "status": "ready",
        }]),
    ], ignore_index=True)
else:
    # Empty configs let the following tuning cells run safely without doing expensive work.
    hist_gradient_tuning_configs = []
    catboost_tuning_configs = []
    tuning_grid_summary = pd.DataFrame([{
        "model": "Manual tuning",
        "grid_configurations": 0,
        "status": "skipped",
    }])

tuning_grid_summary


#### Run Tuning Grid
This block converts grid combinations into model configs, evaluates them on the fixed split, logs them, and keeps the best row per model family.


In [ ]:
# Convert tuning parameters into model configurations only when manual tuning is enabled.
if RUN_MANUAL_TUNING:
    tuning_model_configs = []

    for params in hist_gradient_tuning_configs:
        tuning_model_configs.append({
            "model": "HistGradientBoosting",
            "estimator": HistGradientBoostingRegressor(
                **params,
                random_state=RANDOM_STATE,
            ),
            "hyperparameters": params,
        })

    for params in catboost_tuning_configs:
        tuning_model_configs.append({
            "model": "CatBoost",
            "estimator": CatBoostRegressor(
                **params,
                loss_function="RMSE",
                eval_metric="MAE",
                random_seed=RANDOM_STATE,
                verbose=False,
                allow_writing_files=False,
            ),
            "hyperparameters": params,
        })

    # Evaluate and then append the tuning runs to the same CSV log.
    tuning_results, fitted_tuned_models, tuning_model_log_rows = evaluate_model_configs(
        tuning_model_configs,
        MODEL_TRAIN_INDEX,
        MODEL_TEST_INDEX,
    )
    log_model_runs(tuning_model_log_rows)

    # Rank by lower MAE/RMSE and higher R2 as the final tie-breaker.
    tuning_results_ranked = tuning_results.sort_values(
        by=["model", "MAE", "RMSE", "R2"],
        ascending=[True, True, True, False],
    ).reset_index(drop=True)
    best_tuning_results = (
        tuning_results_ranked
        .groupby("model", group_keys=False)
        .head(1)
        .reset_index(drop=True)
    )
else:
    # Placeholders keep downstream tuning-display cells safe during normal runs.
    tuning_model_configs = []
    tuning_results = pd.DataFrame(columns=[
        "model",
        "n_numeric_features",
        "n_categorical_features",
        "hyperparameters",
        "MAE",
        "RMSE",
        "R2",
    ])
    fitted_tuned_models = {}
    tuning_model_log_rows = []
    tuning_results_ranked = tuning_results.copy()
    best_tuning_results = tuning_results.copy()
    print("Skipping manual tuning because RUN_MANUAL_TUNING is False. Using active_model_configs instead.")

best_tuning_results.round({"MAE": 3, "RMSE": 3, "R2": 3})


#### Top Tuning Candidates
This view keeps the best candidates readable by showing the top rows per model with full hyperparameter strings.


In [ ]:
# Show top tuning candidates only when manual tuning has actually run.
if RUN_MANUAL_TUNING and not tuning_results_ranked.empty:
    # Temporarily disable column truncation so long hyperparameter labels remain readable.
    with pd.option_context("display.max_colwidth", None):
        top_tuning_results = (
            tuning_results_ranked
            .groupby("model", group_keys=False)
            .head(20)
            .reset_index(drop=True)
        )
        display(top_tuning_results.round({"MAE": 3, "RMSE": 3, "R2": 3}))
else:
    top_tuning_results = pd.DataFrame(columns=["model", "hyperparameters", "MAE", "RMSE", "R2"])
    print("No tuning candidates to display because RUN_MANUAL_TUNING is False.")

top_tuning_results[["model", "hyperparameters", "MAE", "RMSE", "R2"]].head(10)


### 3.6 Final evaluation

The final comparison uses three metrics:

- **MAE**: average absolute prediction error in rating points. This is the clearest metric for the project, and lower is better.
- **RMSE**: similar to MAE, but it penalizes larger errors more strongly. Lower is better.
- **R2**: share of target variance explained by the model. Higher is better.

The final table uses the active configurations from Section 3.4. This means Section 3.5 can be skipped once the selected tuned parameters have been copied into `active_model_configs`.


#### Final Ranked Table
This table ranks only the active configurations from the normal notebook run, independent of the optional tuning grid.


In [ ]:
# Build the final ranked table from the active model configurations.
# This cell does not depend on the optional tuning cells above.
final_model_results = (
    model_results
    .sort_values("MAE")
    .reset_index(drop=True)
)

final_model_results.round({"MAE": 3, "RMSE": 3, "R2": 3})


#### Best Final Model
This prints the model with the lowest MAE in the active configuration table.


In [ ]:
# Print the best model in the final ranked table.
best_final_model = final_model_results.iloc[0]

print(
    f"Best final model: {best_final_model['model']} "
    f"with MAE = {best_final_model['MAE']:.3f}"
)


#### Interpretation

Linear regression is expected to be weaker because it mainly learns additive linear effects after scaling and one-hot encoding. Goodreads ratings are likely affected by non-linear relationships between popularity, publication timing, page count, review activity, author metadata, and publisher metadata.

`HistGradientBoostingRegressor` can capture non-linear numeric patterns, so it should usually improve over the linear reference. `CatBoostRegressor` is expected to perform best because it can use high-cardinality categorical book metadata such as `first_author` and `publisher` directly, instead of dropping those columns or turning them into a very wide one-hot encoded matrix.

**[Decision]** Use the best MAE model from the final ranked table as the selected modelling result for this notebook.
